# Cleaning3
## Inizializzazione ed Import

In [8]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

client = DatalakeClient()

# Get the files 
Exclusevily from ADNI dataset stored in the Datalake

In [9]:
file_codes = ['BAIPETNMRCFTP',
'AMPRION_ASYN_SAA',
'PLASMA_ABETA_PROJECT_ADX_VUMC',
'FNIH_PLASMA_PTAU_PROJECT',
'CSFALPHASYN',
'BLENNOWCSFNFL',
'UPENNPLASMA',
'BLENNOWPLASMANFL',
'BLENNOWPLASMATAU',
'C2N_PRECIVITYAD2_PLASMA',
'UGOTPTAU181',
'UPENN_PLASMA_FUJIREBIO_QUANTERIX',
'UCBERKELEY_AMY_6MM',
'UCBERKELEY_TAUPVC_6MM']

In [10]:
search = client.query_files(
    query={'custom.level' : 'cleaned_02', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

## Operazioni
- Trasformare i volumi come percentuali di ICV

In [11]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned2BB'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned3BB'

In [12]:
if os.path.isfile(new_name+'.xlsx'):
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)

The ADNI_variables_cleaned3BB file has been created.
Open the file and verify it, if needed update the variables names and metadata


In [13]:
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')
new_support_file = pd.read_excel(new_name+'.xlsx')

In [14]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    #new_support_file = dataCleaner.update_self_support_file(new_support_file)
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(df_new, file_code)
        # Transform volumes as ICV percentage
        final_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    else:
        final_df = df_new
    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_03', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)

    # Create new file name for datalake
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_03')


    #upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )
   
    
save_df(df_to_save=new_support_file, output_path=new_name)     

RID  non ha valori
usata scappatoia
EXAMDATE  non ha valori
usata scappatoia
VISCODE  non ha valori
usata scappatoia
VISIT_MONTH  non ha valori
usata scappatoia
Amprion_Result  non ha valori
usata scappatoia
METHOD  non ha valori
usata scappatoia
Npositive  non ha valori
usata scappatoia
VISIT_MONTH  non ha valori
usata scappatoia
RID  non ha valori
usata scappatoia
EXAMDATE  non ha valori
usata scappatoia
COHORT  non ha valori
usata scappatoia
ALPHA_SYN  non ha valori
usata scappatoia
METHOD  non ha valori
usata scappatoia
Npositive  non ha valori
usata scappatoia
RID  non ha valori
usata scappatoia
VISCODE  non ha valori
usata scappatoia
VISIT_MONTH  non ha valori
usata scappatoia
EXAMDATE  non ha valori
usata scappatoia
NFL_CSF  non ha valori
usata scappatoia
METHOD  non ha valori
usata scappatoia
Npositive  non ha valori
usata scappatoia
RID  non ha valori
usata scappatoia
VISCODE  non ha valori
usata scappatoia
VISIT_MONTH  non ha valori
usata scappatoia
EXAMDATE  non ha valori
us

In [15]:
final_df

,LONIUID,PTID,RID,VISCODE2,SCANDATE,SITEID,PROCESSDATE,TRACER,TRACER_SUVR_WARNING,META_TEMPORAL_SUVR,...,RIGHT_PUTAMEN_VOLUME,RIGHT_THALAMUS_PROPER_SUVR,RIGHT_THALAMUS_PROPER_VOLUME,RIGHT_VENTRALDC_SUVR,RIGHT_VENTRALDC_VOLUME,RIGHT_VESSEL_SUVR,RIGHT_VESSEL_VOLUME,Apositive,Tpositive,Npositive
0,1594604,011_S_0021,21,m144,2018-02-02,11,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.239,...,4225.0,1.447,5781.0,1.399,3422.0,1.315,14.0,1.0,1.0,0.0
1,1596177,023_S_0031,31,m150,2018-04-24,23,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.167,...,5712.0,1.197,6261.0,1.325,2959.0,1.547,6.0,1.0,1.0,0.0
2,1596172,023_S_0031,31,m162,2019-04-23,23,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.139,...,5967.0,1.134,5947.0,1.241,3008.0,1.438,8.0,1.0,1.0,0.0
3,1598898,067_S_0056,56,m144,2018-02-20,67,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.275,...,4024.0,1.398,6595.0,1.431,3646.0,1.584,42.0,1.0,1.0,0.0
4,1598985,067_S_0056,56,m156,2019-01-10,67,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.275,...,4155.0,1.455,6386.0,1.474,3681.0,1.433,39.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1511,11163538,082_S_7117,7117,m24,2025-03-12,82,2025-07-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.226,...,4560.0,1.251,7364.0,1.323,3663.0,1.136,10.0,1.0,0.0,0.0
1512,11338926,035_S_7121,7121,m24,2025-06-16,35,2025-08-19,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.330,...,4522.0,1.343,7838.0,1.534,3864.0,1.602,27.0,1.0,1.0,0.0
1513,11421729,010_S_6748,6748,m72,2025-07-22,10,2025-09-11,MK6240,DO NOT COMPARE SUVRs ACROSS TRACERS,1.080,...,3871.0,0.700,6147.0,0.921,3271.0,1.008,4.0,0.0,0.0,0.0
1514,11342491,031_S_4021,4021,m168,2025-06-30,31,2025-08-18,PI2620,DO NOT COMPARE SUVRs ACROSS TRACERS,0.999,...,4709.0,0.857,7079.0,0.786,4624.0,0.994,6.0,0.0,0.0,0.0


In [16]:
updated_metadata

{'cofattori': [],
 'file_code': 'UCBERKELEY_TAUPVC_6MM',
 'level': 'cleaned_03',
 'norm_intervallo': ['CSF_SUVR',
  'CTX_ENTORHINAL_VOLUME',
  'CSF_VOLUME',
  'CTX_FUSIFORM_VOLUME',
  'CTX_INFERIORPARIETAL_VOLUME',
  'CTX_INFERIORTEMPORAL_VOLUME',
  'CTX_LATERALOCCIPITAL_VOLUME',
  'CTX_MIDDLETEMPORAL_VOLUME',
  'CTX_PARAHIPPOCAMPAL_VOLUME',
  'CTX_PRECUNEUS_VOLUME'],
 'norm_scala': [],
 'norm_scale_value': {'CSF_SUVR': {'FTP': [0.1678, 1.3657, 'increasing'],
   'unknown': [0.0125, 1.3087, 'increasing']}},
 'norm_volume': [],
 'population': ['ADNI4', 'ADNI3', 'ADNI2'],
 'predittori': ['CSF_SUVR',
  'CTX_ENTORHINAL_VOLUME',
  'CSF_VOLUME',
  'CTX_FUSIFORM_VOLUME',
  'CTX_INFERIORPARIETAL_VOLUME',
  'CTX_INFERIORTEMPORAL_VOLUME',
  'CTX_LATERALOCCIPITAL_VOLUME',
  'CTX_MIDDLETEMPORAL_VOLUME',
  'CTX_PARAHIPPOCAMPAL_VOLUME',
  'CTX_PRECUNEUS_VOLUME'],
 'source': 'ADNI',
 'volume_norm_values': []}